# Step 2: Parse Filings & Extract Text

In this notebook we:
1. Locate the downloaded HTML filings from Step 1
2. Extract clean text using BeautifulSoup (free, no API key needed)
3. Handle financial tables separately
4. Save extracted text as `.txt` files ready for chunking in Step 3

**Why HTML and not PDF?**  
SEC EDGAR publishes filings in HTML format. This is actually easier to parse than PDF — 
we get structured text with proper headings and tables, rather than fighting PDF layout issues.

## 2.1 Install Dependencies

In [12]:
!pip install beautifulsoup4 lxml pandas

## 2.2 Imports & Configuration

In [13]:
import os
import re
import glob
import json
from bs4 import BeautifulSoup
import pandas as pd

# Paths — should match Step 1
SEC_FILINGS_DIR = os.path.join("data", "sec_filings", "sec-edgar-filings")
OUTPUT_DIR = os.path.join("data", "extracted_texts")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Company config — must match Step 1
COMPANIES = [
    ("MSFT",  "10-K", "Microsoft"),
    ("GOOGL", "10-K", "Alphabet (Google)"),
    ("ACN",   "10-K", "Accenture"),
    ("IBM",   "10-K", "IBM"),
    ("CTSH",  "10-K", "Cognizant"),
    ("INFY",  "20-F", "Infosys"),
    ("WIT",   "20-F", "Wipro"),
]

print(f"Looking for filings in: {os.path.abspath(SEC_FILINGS_DIR)}")
print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")

Looking for filings in: c:\Users\anfas\Downloads\finance report rag\data\sec_filings\sec-edgar-filings
Output directory: c:\Users\anfas\Downloads\finance report rag\data\extracted_texts


## 2.3 Locate and Extract the Primary Filing from full-submission.txt

The `sec-edgar-downloader` saves each filing as a single `full-submission.txt` file.  
This is an SGML bundle that contains the actual HTML filing plus all exhibits.  

We need to:
1. Open `full-submission.txt`
2. Find the primary document (the one with `<TYPE>10-K` or `<TYPE>20-F`)
3. Extract its HTML content for parsing

In [14]:
def extract_primary_html_from_submission(submission_path, form_type):
    """
    Parse full-submission.txt (SGML format) and extract the primary
    HTML document.
    
    The file contains multiple <DOCUMENT>...</DOCUMENT> blocks.
    We find the one whose <TYPE> matches the form_type (10-K or 20-F).
    If no exact match, we fall back to the first and largest document.
    
    Returns: (html_content_string, filename) or (None, None)
    """
    with open(submission_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    
    # Split into individual documents
    doc_pattern = re.compile(r'<DOCUMENT>(.*?)</DOCUMENT>', re.DOTALL)
    documents = doc_pattern.findall(content)
    
    if not documents:
        return None, None
    
    best_doc = None
    best_filename = None
    
    for doc in documents:
        # Extract the TYPE tag
        type_match = re.search(r'<TYPE>(.+)', doc)
        doc_type = type_match.group(1).strip() if type_match else ''
        
        # Extract the FILENAME tag
        fname_match = re.search(r'<FILENAME>(.+)', doc)
        doc_filename = fname_match.group(1).strip() if fname_match else 'unknown'
        
        # Check if this is the primary filing document
        # Match the form type (e.g., '10-K' or '20-F')
        if doc_type.upper() == form_type.upper():
            # Extract the HTML portion (everything in <TEXT>...</TEXT>)
            text_match = re.search(r'<TEXT>(.*?)</TEXT>', doc, re.DOTALL)
            if text_match:
                best_doc = text_match.group(1)
                best_filename = doc_filename
                break  # Found the primary document
    
    # Fallback: if no exact type match, use the first .htm document
    if best_doc is None:
        for doc in documents:
            fname_match = re.search(r'<FILENAME>(.+)', doc)
            doc_filename = fname_match.group(1).strip() if fname_match else ''
            
            if doc_filename.lower().endswith(('.htm', '.html')):
                text_match = re.search(r'<TEXT>(.*?)</TEXT>', doc, re.DOTALL)
                if text_match:
                    html_content = text_match.group(1)
                    if best_doc is None or len(html_content) > len(best_doc):
                        best_doc = html_content
                        best_filename = doc_filename
    
    return best_doc, best_filename


# Locate and extract primary documents for all companies
filing_data = {}  # ticker -> (html_content, filename, submission_path)

for ticker, form_type, name in COMPANIES:
    ticker_dir = os.path.join(SEC_FILINGS_DIR, ticker, form_type)
    
    if not os.path.exists(ticker_dir):
        print(f"  ✗ {ticker} — directory not found: {ticker_dir}")
        continue
    
    # Get the filing subfolder (named by accession number)
    subfolders = [f for f in os.listdir(ticker_dir) 
                  if os.path.isdir(os.path.join(ticker_dir, f))]
    
    if not subfolders:
        print(f"  ✗ {ticker} — no filing subfolders found")
        continue
    
    # Path to full-submission.txt
    submission_path = os.path.join(ticker_dir, subfolders[0], 'full-submission.txt')
    
    if not os.path.exists(submission_path):
        print(f"  ✗ {ticker} — full-submission.txt not found")
        continue
    
    print(f"  Processing {ticker} ({name})...")
    html_content, doc_filename = extract_primary_html_from_submission(submission_path, form_type)
    
    if html_content:
        html_size_kb = len(html_content) / 1024
        filing_data[ticker] = (html_content, doc_filename, submission_path)
        print(f"  ✓ {ticker} — extracted '{doc_filename}' ({html_size_kb:.1f} KB of HTML)")
    else:
        print(f"  ✗ {ticker} — could not extract primary document")

print(f"\nExtracted {len(filing_data)} primary documents out of {len(COMPANIES)} companies")

  Processing MSFT (Microsoft)...
  ✓ MSFT — extracted 'msft-20250630.htm' (7966.9 KB of HTML)
  Processing GOOGL (Alphabet (Google))...
  ✓ GOOGL — extracted 'goog-20251231.htm' (2555.2 KB of HTML)
  Processing ACN (Accenture)...
  ✓ ACN — extracted 'acn-20250831.htm' (2753.2 KB of HTML)
  Processing IBM (IBM)...
  ✓ IBM — extracted 'ibm-20251231.htm' (1051.9 KB of HTML)
  Processing CTSH (Cognizant)...
  ✓ CTSH — extracted 'ctsh-20251231.htm' (2625.0 KB of HTML)
  Processing INFY (Infosys)...
  ✓ INFY — extracted 'infy-20260331.htm' (12755.7 KB of HTML)
  Processing WIT (Wipro)...
  ✓ WIT — extracted 'wit-20260331.htm' (14279.1 KB of HTML)

Extracted 7 primary documents out of 7 companies


## 2.4 Text Extraction Functions

We use BeautifulSoup to:
1. Strip all HTML tags and extract readable text
2. Convert HTML tables into a structured text format (pipe-separated)
3. Clean up whitespace and encoding artifacts

**Important caveat:** SEC filings are messy — they have inline XBRL tags, inconsistent 
formatting, and deeply nested HTML. Our extraction won't be perfect, but it will be 
good enough for RAG retrieval. Perfect extraction is a hard engineering problem 
that paid APIs like sec-api.io solve — but we're keeping this free.

In [15]:
def extract_tables_as_text(soup):
    """
    Find all HTML tables and convert them to pipe-separated text.
    Returns a list of (table_element, table_as_text) tuples.
    """
    table_texts = []
    
    for table in soup.find_all('table'):
        rows = []
        for tr in table.find_all('tr'):
            cells = []
            for td in tr.find_all(['td', 'th']):
                cell_text = td.get_text(separator=' ', strip=True)
                # Clean up common artifacts
                cell_text = re.sub(r'\s+', ' ', cell_text)
                cells.append(cell_text)
            if any(c.strip() for c in cells):  # Skip entirely empty rows
                rows.append(' | '.join(cells))
        
        if rows:
            table_text = '\n'.join(rows)
            table_texts.append((table, table_text))
    
    return table_texts


def clean_text(text):
    """
    Clean extracted text:
    - Remove excessive whitespace and blank lines
    - Remove common HTML/XBRL artifacts
    - Normalize unicode characters
    """
    # Replace common unicode characters
    text = text.replace('\xa0', ' ')   # non-breaking space
    text = text.replace('\u200b', '')  # zero-width space
    text = text.replace('\u2019', "'") # right single quote
    text = text.replace('\u2018', "'") # left single quote
    text = text.replace('\u201c', '"') # left double quote
    text = text.replace('\u201d', '"') # right double quote
    text = text.replace('\u2014', ' - ') # em dash
    text = text.replace('\u2013', ' - ') # en dash
    
    # Collapse multiple spaces into one
    text = re.sub(r'[ \t]+', ' ', text)
    
    # Collapse 3+ consecutive newlines into 2
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    # Remove lines that are only whitespace
    lines = text.split('\n')
    lines = [line.strip() for line in lines]
    text = '\n'.join(lines)
    
    # Collapse again after stripping
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    return text.strip()


def extract_text_from_html_string(html_string):
    """
    Extract clean text from an HTML string (extracted from full-submission.txt).
    
    Strategy:
    1. Parse HTML with BeautifulSoup
    2. Remove script, style, and XBRL-related tags
    3. Convert tables to pipe-separated text and replace in-place
    4. Extract remaining text
    5. Clean up the result
    
    Returns: (clean_text, metadata_dict)
    """
    soup = BeautifulSoup(html_string, 'lxml')
    
    # Remove elements that don't contain useful text
    for tag_name in ['script', 'style', 'meta', 'link', 'header', 'footer']:
        for tag in soup.find_all(tag_name):
            tag.decompose()
    
    # Remove XBRL/iXBRL processing tags (keep their text content)
    for tag in soup.find_all(re.compile(r'^(ix:|xbrl)', re.IGNORECASE)):
        tag.unwrap()  # Keeps text, removes the tag
    
    # Convert tables to text and replace them
    table_texts = extract_tables_as_text(soup)
    for table_element, table_text in table_texts:
        table_marker = soup.new_string(f'\n\n[TABLE]\n{table_text}\n[/TABLE]\n\n')
        table_element.replace_with(table_marker)
    
    # Extract all remaining text
    text = soup.get_text(separator='\n')
    
    # Clean it up
    text = clean_text(text)
    
    # Gather basic metadata
    metadata = {
        'html_input_length': len(html_string),
        'extracted_text_length': len(text),
        'num_tables_found': len(table_texts),
    }
    
    return text, metadata

print("Extraction functions defined.")

Extraction functions defined.


## 2.5 Run Extraction on All Filings

This processes each filing and saves the extracted text as a `.txt` file.

In [16]:
extraction_results = []

for ticker, form_type, name in COMPANIES:
    if ticker not in filing_data:
        print(f"  ✗ {ticker} — skipped (no filing extracted)")
        continue
    
    html_content, doc_filename, submission_path = filing_data[ticker]
    print(f"\nProcessing {ticker} ({name})...")
    
    try:
        text, metadata = extract_text_from_html_string(html_content)
        
        # Add company info to metadata
        metadata['ticker'] = ticker
        metadata['company_name'] = name
        metadata['form_type'] = form_type
        metadata['source_file'] = doc_filename
        metadata['submission_path'] = submission_path
        
        # Save extracted text
        output_filename = f"{ticker}_{form_type}_extracted.txt"
        output_path = os.path.join(OUTPUT_DIR, output_filename)
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(text)
        
        metadata['output_file'] = output_filename
        extraction_results.append(metadata)
        
        text_len_k = metadata['extracted_text_length'] / 1000
        print(f"  ✓ {ticker} — {text_len_k:.1f}K chars, {metadata['num_tables_found']} tables")
        print(f"    Saved to: {output_path}")
        
    except Exception as e:
        print(f"  ✗ {ticker} — ERROR: {e}")

print(f"\nExtracted text for {len(extraction_results)}/{len(COMPANIES)} companies")


Processing MSFT (Microsoft)...
  ✓ MSFT — 369.4K chars, 85 tables
    Saved to: data\extracted_texts\MSFT_10-K_extracted.txt

Processing GOOGL (Alphabet (Google))...
  ✓ GOOGL — 401.4K chars, 181 tables
    Saved to: data\extracted_texts\GOOGL_10-K_extracted.txt

Processing ACN (Accenture)...
  ✓ ACN — 407.8K chars, 188 tables
    Saved to: data\extracted_texts\ACN_10-K_extracted.txt

Processing IBM (IBM)...
  ✓ IBM — 213.6K chars, 19 tables
    Saved to: data\extracted_texts\IBM_10-K_extracted.txt

Processing CTSH (Cognizant)...
  ✓ CTSH — 388.1K chars, 215 tables
    Saved to: data\extracted_texts\CTSH_10-K_extracted.txt

Processing INFY (Infosys)...
  ✓ INFY — 1040.6K chars, 191 tables
    Saved to: data\extracted_texts\INFY_20-F_extracted.txt

Processing WIT (Wipro)...
  ✓ WIT — 945.6K chars, 156 tables
    Saved to: data\extracted_texts\WIT_20-F_extracted.txt

Extracted text for 7/7 companies


## 2.6 Extraction Summary

In [17]:
if extraction_results:
    df_summary = pd.DataFrame(extraction_results)
    df_summary['text_length_K'] = (df_summary['extracted_text_length'] / 1000).round(1)
    df_summary['file_size_MB'] = (df_summary['file_size_bytes'] / (1024*1024)).round(2)
    
    display_cols = ['ticker', 'company_name', 'form_type', 'file_size_MB', 
                    'text_length_K', 'num_tables_found', 'output_file']
    print(df_summary[display_cols].to_string(index=False))
    
    print(f"\nTotal extracted text: {df_summary['text_length_K'].sum():.1f}K characters")
    print(f"Total tables found: {df_summary['num_tables_found'].sum()}")
    
    # Save metadata for reference
    metadata_path = os.path.join(OUTPUT_DIR, 'extraction_metadata.json')
    with open(metadata_path, 'w') as f:
        json.dump(extraction_results, f, indent=2)
    print(f"\nMetadata saved to: {metadata_path}")
else:
    print("No extractions completed. Check that Step 1 downloaded files correctly.")

KeyError: 'file_size_bytes'

## 2.7 Preview Extracted Text

Let's look at what we extracted to make sure it's usable.

In [ ]:
# Preview the first successfully extracted file
if extraction_results:
    preview_file = os.path.join(OUTPUT_DIR, extraction_results[0]['output_file'])
    ticker = extraction_results[0]['ticker']
    
    with open(preview_file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    print(f"=== Preview: {ticker} (first 2000 chars) ===")
    print(content[:2000])
    print("\n... [truncated] ...")
else:
    print("No files to preview.")

In [ ]:
# Preview a table section to verify table extraction
if extraction_results:
    preview_file = os.path.join(OUTPUT_DIR, extraction_results[0]['output_file'])
    
    with open(preview_file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Find the first [TABLE] block
    table_start = content.find('[TABLE]')
    if table_start != -1:
        table_end = content.find('[/TABLE]', table_start)
        if table_end != -1:
            table_text = content[table_start:table_end + len('[/TABLE]')]
            # Show first 1000 chars of the table
            print(f"=== First table found in {extraction_results[0]['ticker']} ===")
            print(table_text[:1000])
            if len(table_text) > 1000:
                print("\n... [table truncated] ...")
    else:
        print("No [TABLE] markers found in the extracted text.")
else:
    print("No files to preview.")

## 2.8 Quick Quality Check

Let's verify the extracted text contains expected financial sections.  
A good 10-K extraction should contain keywords like "Risk Factors", "Revenue", "Net Income", etc.

In [ ]:
# Financial keywords we expect to find in a well-extracted 10-K
EXPECTED_KEYWORDS = [
    'risk factors',
    'revenue',
    'net income',
    'cash flow',
    'total assets',
    'management discussion',  # MD&A section
    'financial statements',
]

print(f"{'Ticker':<8} {'Size (K chars)':<16} {'Keywords Found':<16} {'Missing Keywords'}")
print("-" * 70)

for result in extraction_results:
    filepath = os.path.join(OUTPUT_DIR, result['output_file'])
    with open(filepath, 'r', encoding='utf-8') as f:
        text_lower = f.read().lower()
    
    found = [kw for kw in EXPECTED_KEYWORDS if kw in text_lower]
    missing = [kw for kw in EXPECTED_KEYWORDS if kw not in text_lower]
    text_len_k = result['extracted_text_length'] / 1000
    
    missing_str = ', '.join(missing) if missing else 'None'
    print(f"{result['ticker']:<8} {text_len_k:<16.1f} {len(found)}/{len(EXPECTED_KEYWORDS):<14} {missing_str}")

---

## ✅ Step 2 Complete

**What we did:**
- Located the primary HTML document in each filing folder
- Extracted clean text using BeautifulSoup, handling:
  - HTML tag removal
  - XBRL/iXBRL tag cleanup
  - Table conversion to pipe-separated text format
  - Unicode and whitespace normalization
- Ran a quality check to verify expected financial keywords are present

**Output files:**
- `data/extracted_texts/{TICKER}_{FORM_TYPE}_extracted.txt` — one per company
- `data/extracted_texts/extraction_metadata.json` — extraction stats

**Known limitations:**
- Table extraction is approximate — complex nested tables may not render perfectly
- Some XBRL artifacts may remain in the text
- Section boundaries are not explicitly identified (we handle this in Step 3 via chunking)

**Next step:** `Step3_Chunking_and_Embedding.ipynb` — split the text into chunks and store them in a vector database